# Actividad 4 — Comparación de modelos preentrenados para análisis de sentimiento
**Caso de uso:** clasificación de reseñas de restaurante en español (positivo, neutral, negativo).

Este notebook compara tres modelos de Hugging Face con un dataset pequeño, balanceado y **sintético**. Los resultados miden el comportamiento en este conjunto de prueba, no representan una garantía de desempeño en reseñas reales. Para una decisión de producción se necesita una muestra real, representativa y revisada por personas.

**Reproducibilidad:** ejecutar las celdas en orden en Google Colab. Seleccionar `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU`. Las métricas y latencias se calculan al ejecutar; no se incluyen números inventados.


## 1. Configuración de GPU y dependencias
Se verifica el acelerador disponible. En Colab, selecciona una GPU antes de continuar. Si no hay GPU, el notebook seguirá en CPU, pero las latencias no serán comparables con una ejecución en GPU.


In [ ]:
!nvidia-smi

In [ ]:
%pip -q install -U transformers accelerate evaluate scikit-learn pandas matplotlib huggingface_hub

## 2. Autenticación opcional con Hugging Face Hub
Los tres modelos son públicos y normalmente pueden descargarse sin token. La autenticación se incluye para cumplir el flujo de trabajo del tema y permitir acceso a recursos con permisos. Guarda el token en `Colab Secrets` como `HF_TOKEN`; no lo escribas directamente en el notebook ni lo subas a GitHub.


In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None

if not token:
    print("No se encontró HF_TOKEN en Colab Secrets. Para modelos públicos, puedes continuar sin autenticarte.")
else:
    from huggingface_hub import login
    login(token=token, add_to_git_credential=False)
    print("Autenticación realizada sin mostrar el token.")


## 3. Carga del dataset y revisión
El archivo `data/resenas_restaurante.csv` contiene 90 ejemplos creados específicamente para esta práctica: 30 positivos, 30 negativos y 30 neutrales. La fuente es un script incluido en `src/generar_dataset.py`; no son opiniones reales de clientes.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

DATA_PATH = Path("/content/resenas_restaurante.csv")
if not DATA_PATH.exists():
    # Si el notebook está abierto desde una carpeta local o repositorio, intenta rutas habituales.
    candidates = [Path("data/resenas_restaurante.csv"), Path("../data/resenas_restaurante.csv")]
    for candidate in candidates:
        if candidate.exists():
            DATA_PATH = candidate
            break

if not DATA_PATH.exists():
    from google.colab import files
    uploaded = files.upload()  # seleccionar resenas_restaurante.csv
    DATA_PATH = Path("resenas_restaurante.csv")

df = pd.read_csv(DATA_PATH)
required = {"texto", "etiqueta"}
assert required.issubset(df.columns), f"Faltan columnas requeridas: {required - set(df.columns)}"
df = df.dropna(subset=["texto", "etiqueta"]).copy()
df["etiqueta"] = df["etiqueta"].str.upper().str.strip()
df = df[df["etiqueta"].isin(["POS", "NEG", "NEU"])].reset_index(drop=True)
display(df.head())
display(df["etiqueta"].value_counts().rename_axis("etiqueta").to_frame("registros"))
print(f"Registros evaluables: {len(df)}")


## 4. Criterios de selección
- Compatibilidad con español y salida de sentimiento.
- Capacidad de mapear las etiquetas a POS/NEG/NEU.
- Latencia promedio por reseña.
- Métricas macro para evitar que una clase domine la comparación.
- Tamaño aproximado, licencia y dominio de entrenamiento, revisados en las fichas de los modelos.

**Advertencia metodológica:** el conjunto sintético es pequeño y sencillo; sus métricas sirven para comprobar el pipeline y comparar esta muestra, pero no son un benchmark independiente ni representativo de clientes reales.


In [ ]:
from transformers import pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import time, gc, torch


In [ ]:
MODEL_IDS = [
    "pysentimiento/robertuito-sentiment-analysis",
    "lxyuan/distilbert-base-multilingual-cased-sentiments-student",
    "cardiffnlp/twitter-xlm-roberta-base-sentiment",
]

# Normaliza nombres distintos de las salidas de los modelos.
def normalize_label(label):
    x = str(label).strip().lower()
    if x in {"pos", "positive", "label_2", "5 stars", "4 stars"} or "positive" in x:
        return "POS"
    if x in {"neg", "negative", "label_0", "1 star", "2 stars"} or "negative" in x:
        return "NEG"
    if x in {"neu", "neutral", "label_1", "3 stars"} or "neutral" in x:
        return "NEU"
    # Etiquetas habituales de RoBERTuito
    if x == "pos": return "POS"
    if x == "neg": return "NEG"
    if x == "neu": return "NEU"
    raise ValueError(f"Etiqueta no reconocida: {label!r}. Revisa config.id2label del modelo.")

def predict_one(pipe, text):
    result = pipe(str(text), truncation=True, max_length=256)[0]
    return normalize_label(result["label"]), float(result["score"])


## 5. Inferencia, métricas y latencia
La latencia se mide secuencialmente por texto, tras cargar el modelo. Se excluye la descarga inicial y la carga en memoria; por ello representa latencia de inferencia local aproximada, no el tiempo total de despliegue. En GPU se sincroniza CUDA para medir correctamente.


In [ ]:
results = []
predictions_by_model = {}

device = 0 if torch.cuda.is_available() else -1
print("Dispositivo:", torch.cuda.get_device_name(0) if device == 0 else "CPU")

for model_id in MODEL_IDS:
    print(f"\nCargando {model_id}")
    pipe = pipeline("text-classification", model=model_id, device=device)
    y_true = df["etiqueta"].tolist()
    y_pred, confidences, latencies = [], [], []

    # Calentamiento: no se incluye en latencia
    _ = pipe(str(df.iloc[0]["texto"]), truncation=True, max_length=256)
    if device == 0:
        torch.cuda.synchronize()

    for text in df["texto"].tolist():
        if device == 0:
            torch.cuda.synchronize()
        start = time.perf_counter()
        label, confidence = predict_one(pipe, text)
        if device == 0:
            torch.cuda.synchronize()
        elapsed_ms = (time.perf_counter() - start) * 1000
        y_pred.append(label)
        confidences.append(confidence)
        latencies.append(elapsed_ms)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=["POS", "NEU", "NEG"], average="macro", zero_division=0
    )
    results.append({
        "modelo": model_id,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
        "latencia_media_ms": float(np.mean(latencies)),
        "latencia_p50_ms": float(np.percentile(latencies, 50)),
        "latencia_p95_ms": float(np.percentile(latencies, 95)),
        "confianza_media": float(np.mean(confidences)),
        "dispositivo": "GPU" if device == 0 else "CPU",
        "n_muestras": len(y_true)
    })
    predictions_by_model[model_id] = y_pred
    print(classification_report(y_true, y_pred, labels=["POS","NEU","NEG"], zero_division=0))
    del pipe
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results_df = pd.DataFrame(results).sort_values(["f1_macro", "latencia_media_ms"], ascending=[False, True])
display(results_df)
results_df.to_csv("comparacion_modelos.csv", index=False)


## 6. Comparación y visualización
Se recomienda priorizar F1 macro y revisar por separado los errores por clase. Una latencia menor no compensa necesariamente un recall bajo en una clase crítica.


In [ ]:
import matplotlib.pyplot as plt

display(results_df[["modelo", "accuracy", "precision_macro", "recall_macro", "f1_macro", "latencia_media_ms"]])
ax = results_df.plot(x="modelo", y=["accuracy", "precision_macro", "recall_macro", "f1_macro"], kind="bar", figsize=(12,5))
ax.set_ylim(0, 1)
ax.set_ylabel("Métrica")
ax.set_title("Métricas por modelo (dataset sintético)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

ax = results_df.plot(x="modelo", y="latencia_media_ms", kind="bar", figsize=(10,4), legend=False)
ax.set_ylabel("Milisegundos por reseña")
ax.set_title("Latencia media de inferencia")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


## 7. Matrices de confusión y errores


In [ ]:
for model_id, y_pred in predictions_by_model.items():
    cm = confusion_matrix(df["etiqueta"], y_pred, labels=["POS", "NEU", "NEG"])
    print("\n", model_id)
    display(pd.DataFrame(cm, index=["Real POS","Real NEU","Real NEG"], columns=["Pred POS","Pred NEU","Pred NEG"]))


## 8. Conclusiones parciales y decisión
La recomendación final debe basarse en los valores generados al ejecutar este notebook, no en una predicción previa de cuál modelo ganará. Selecciona el modelo que combine F1 macro suficiente, latencia aceptable y una licencia compatible con el uso previsto.

**Umbrales iniciales propuestos para un prototipo:** F1 macro ≥ 0.80 y latencia media ≤ 250 ms por reseña en el hardware de prueba. Son criterios de proyecto, no resultados obtenidos. Si ninguno los cumple, no debe declararse un ganador de producción: ampliar y validar el dataset, revisar errores y considerar ajustes mínimos o fine-tuning.

**Riesgos:** dataset sintético y pequeño; dominio de entrenamiento basado en redes sociales; sarcasmo, ironía, modismos regionales, reseñas mixtas y sesgo de clase. Antes de producción, evaluar una muestra real etiquetada por humanos, documentar el hardware y repetir la medición varias veces.


In [ ]:
# Exporta las predicciones por reseña para auditoría
audit = df[["id", "texto", "etiqueta"]].copy()
for model_id, preds in predictions_by_model.items():
    audit[model_id.split("/")[-1]] = preds
audit.to_csv("predicciones_por_resena.csv", index=False, encoding="utf-8-sig")
display(audit.head(10))
print("Archivos generados: comparacion_modelos.csv y predicciones_por_resena.csv")


## 9. Referencias
- RoBERTuito sentiment: https://huggingface.co/pysentimiento/robertuito-sentiment-analysis
- DistilBERT multilingual sentiment student: https://huggingface.co/lxyuan/distilbert-base-multilingual-cased-sentiments-student
- CardiffNLP Twitter XLM-RoBERTa sentiment: https://huggingface.co/cardiffnlp/twitter-xlm-roberta-base-sentiment
- Hugging Face Transformers: https://huggingface.co/docs/transformers
- Scikit-learn metrics: https://scikit-learn.org/stable/modules/model_evaluation.html

Revisar las fichas vigentes para confirmar licencia, condiciones de redistribución y limitaciones antes de un uso comercial.
